In [0]:
%sql
CREATE CATALOG IF NOT EXISTS happiness;

CREATE SCHEMA IF NOT EXISTS happiness.landing;
CREATE SCHEMA IF NOT EXISTS happiness.bronze;
CREATE SCHEMA IF NOT EXISTS happiness.silver;
CREATE SCHEMA IF NOT EXISTS happiness.gold;
CREATE SCHEMA IF NOT EXISTS happiness.meta;

-- Volume: almacenamiento para los archivos crudos (xlsx, csv.gz, json) antes de parsearlos
CREATE VOLUME IF NOT EXISTS happiness.landing.archivos;

In [0]:
%sql
WITH fuente_datos AS (
  SELECT * FROM VALUES
    ('ILOSTAT', 'HOW_TEMP_SEX_NB_A',    'Horas semanales habituales por persona ocupada',      'Tiempo de trabajo',  'horas/semana', false, 'cobertura < 80% con arrastre de 2 años', DATE'2026-11-10'),
    ('ILOSTAT', 'EMP_NIFL_SEX_RT_A',    'Tasa de empleo informal',                              'Informalidad',       '%',            false, 'cobertura < 80% con arrastre de 2 años', DATE'2026-11-10'),
    ('ILOSTAT', 'EMP_XTRU_SEX_RT_A',    'Subempleo por insuficiencia de horas',                 'Subempleo',          '%',            false, 'cobertura < 80% con arrastre de 2 años', DATE'2026-11-10'),
    ('ILOSTAT', 'EIP_NEET_SEX_RT_A',    'Jovenes que ni estudian ni trabajan (NEET)',           'Juventud',           '%',            true,  'excepcion D4/D6: 75.5%, cerca del umbral y unico indicador de juventud', DATE'2026-11-10'),
    ('ILOSTAT', 'LUU_2LU2_SEX_RT_A',    'Tasa de subutilizacion de la fuerza laboral (LU2)',    'Calidad del empleo', '%',            true,  'supera 80% de cobertura; compuesto que incluye desempleo y fuerza laboral potencial', DATE'2026-11-10'),
    ('ILOSTAT', 'HOW_TEMP_SEX_AGE_NB_A','Horas semanales por sexo y edad',                      'Tiempo de trabajo',  'horas/semana', false, 'cobertura < 80%; redundante con HOW_TEMP_SEX_NB_A', DATE'2026-11-10')
  AS t(fuente, codigo, nombre, tema, unidad, incluido, motivo, fecha_decision)
)
MERGE INTO happiness.meta.config_indicadores AS destino
USING fuente_datos
ON destino.fuente = fuente_datos.fuente AND destino.codigo = fuente_datos.codigo
WHEN NOT MATCHED THEN INSERT *;

SELECT * FROM happiness.meta.config_indicadores WHERE incluido = true;

In [0]:
# Celda para agregar a 00_setup.ipynb, DESPUES de la celda que crea/carga la tabla
# happiness.meta.config_indicadores (la de ILOSTAT). Carga el resultado de la
# auditoria de Banco Mundial (wb_audit.py) + las 6 dimensiones de WGI que medimos
# directo contra el dataset oficial (wgidataset_with_sourcedata-2026.xlsx).

# Cada tupla: (fuente, codigo, nombre, tema, unidad, incluido, motivo)
# "incluido=False" no se descarta de la tabla: queda como registro de auditoria,
# igual que hicimos con los 4 indicadores de ILOSTAT que no pasaron el corte.
filas_wb = [
    # --- demografia ---
    ("BANCO_MUNDIAL", "SP.DYN.TFRT.IN", "Tasa de fertilidad", "demografia", "nacidos por mujer", True, "cobertura 99.3% con arrastre de 2 anios"),
    ("BANCO_MUNDIAL", "SP.URB.TOTL.IN.ZS", "Poblacion urbana", "demografia", "%", True, "cobertura 99.3% con arrastre de 2 anios"),
    # --- salud ---
    ("BANCO_MUNDIAL", "SP.DYN.LE00.IN", "Esperanza de vida al nacer", "salud", "anios", True, "cobertura 99.3% con arrastre de 2 anios"),
    ("BANCO_MUNDIAL", "SH.DYN.MORT", "Mortalidad menores de 5", "salud", "por 1000 nacidos vivos", True, "cobertura 98.6% con arrastre de 2 anios"),
    ("BANCO_MUNDIAL", "SH.XPD.CHEX.GD.ZS", "Gasto en salud", "salud", "% PIB", True, "cobertura 98.1% con arrastre de 2 anios"),
    ("BANCO_MUNDIAL", "SH.STA.SUIC.P5", "Tasa de suicidio", "salud", "por 100k habitantes", True, "cobertura 84.9% con arrastre de 2 anios"),
    # --- trabajo (incluye servicios/industria, asi lo agrupo el doc de fuentes original) ---
    ("BANCO_MUNDIAL", "SL.TLF.CACT.ZS", "Participacion laboral", "trabajo", "% poblacion 15+", True, "cobertura 99.2% con arrastre de 2 anios"),
    ("BANCO_MUNDIAL", "SL.UEM.TOTL.ZS", "Desempleo total", "trabajo", "% fuerza laboral", True, "cobertura 99.2% con arrastre de 2 anios"),
    ("BANCO_MUNDIAL", "SL.EMP.VULN.ZS", "Empleo vulnerable", "trabajo", "% empleo total", True, "cobertura 99.2% con arrastre de 2 anios"),
    ("BANCO_MUNDIAL", "NV.IND.TOTL.ZS", "Industria", "trabajo", "% PIB", True, "cobertura 97.9% con arrastre de 2 anios"),
    ("BANCO_MUNDIAL", "NV.SRV.TOTL.ZS", "Servicios", "trabajo", "% PIB", True, "cobertura 97.9% con arrastre de 2 anios"),
    # --- economia ---
    ("BANCO_MUNDIAL", "NY.GDP.MKTP.KD.ZG", "Crecimiento del PIB", "economia", "% anual", True, "cobertura 98.9% con arrastre de 2 anios"),
    ("BANCO_MUNDIAL", "NY.GDP.PCAP.PP.KD", "PIB per capita (PPA)", "economia", "USD PPA constantes", True, "cobertura 97.7% con arrastre de 2 anios"),
    ("BANCO_MUNDIAL", "FP.CPI.TOTL.ZG", "Inflacion anual", "economia", "% anual", True, "cobertura 96.8% con arrastre de 2 anios"),
    ("BANCO_MUNDIAL", "NE.TRD.GNFS.ZS", "Comercio", "economia", "% PIB", True, "cobertura 93.4% con arrastre de 2 anios"),
    ("BANCO_MUNDIAL", "SI.POV.GINI", "Indice de Gini", "economia", "indice 0-100", True, "excepcion D4: cobertura 65.7%, central para la hipotesis"),
    # --- ambiente ---
    ("BANCO_MUNDIAL", "EN.ATM.PM25.MC.M3", "Exposicion a PM2.5", "ambiente", "ug/m3", True, "cobertura 98.7% con arrastre de 2 anios"),
    ("BANCO_MUNDIAL", "AG.LND.FRST.ZS", "Area forestal", "ambiente", "% territorio", True, "cobertura 98.6% con arrastre de 2 anios"),
    ("BANCO_MUNDIAL", "EN.GHG.CO2.PC.CE.AR5", "Emisiones de CO2 per capita", "ambiente", "toneladas per capita", True, "cobertura 97.1% con arrastre de 2 anios"),
    ("BANCO_MUNDIAL", "EG.FEC.RNEW.ZS", "Energia renovable", "ambiente", "% consumo energia", True, "cobertura 86.4% con arrastre de 2 anios"),
    ("BANCO_MUNDIAL", "EN.ATM.CO2E.PC", "Emisiones de CO2 per capita (codigo viejo)", "ambiente", "toneladas per capita", False, "codigo descontinuado por el Banco Mundial; reemplazado por EN.GHG.CO2.PC.CE.AR5"),
    # --- tecnologia ---
    ("BANCO_MUNDIAL", "IT.NET.USER.ZS", "Usuarios de internet", "tecnologia", "% poblacion", True, "cobertura 98.7% con arrastre de 2 anios"),
    ("BANCO_MUNDIAL", "IT.CEL.SETS.P2", "Suscripciones celular", "tecnologia", "por 100 personas", True, "cobertura 99.0% con arrastre de 2 anios"),
    ("BANCO_MUNDIAL", "IT.NET.BBND.P2", "Banda ancha fija", "tecnologia", "por 100 personas", True, "cobertura 98.0% con arrastre de 2 anios"),
    # --- seguridad ---
    ("BANCO_MUNDIAL", "MS.MIL.XPND.GD.ZS", "Gasto militar", "seguridad", "% PIB", True, "cobertura 92.9% con arrastre de 2 anios"),
    ("BANCO_MUNDIAL", "SG.GEN.PARL.ZS", "Mujeres en el parlamento", "seguridad", "% bancas", True, "cobertura 97.5% con arrastre de 2 anios"),
    ("BANCO_MUNDIAL", "VC.IHR.PSRC.P5", "Homicidios intencionales", "seguridad", "por 100k habitantes", True, "excepcion D4: cobertura 73.3%, central para la hipotesis"),
    # --- gobernanza (WGI, dataset oficial descargado: wgidataset_with_sourcedata-2026.xlsx) ---
    ("WGI", "va", "Voz y rendicion de cuentas", "gobernanza", "estimador -2.5 a 2.5", True, "cobertura 100%; conservada, no se solapa con las otras 3 elegidas"),
    ("WGI", "pv", "Estabilidad politica", "gobernanza", "estimador -2.5 a 2.5", True, "cobertura 100%; conservada, no se solapa con las otras 3 elegidas"),
    ("WGI", "ge", "Efectividad del gobierno", "gobernanza", "estimador -2.5 a 2.5", True, "cobertura 100%; preferida sobre rq (Spearman 0.924 entre ambas)"),
    ("WGI", "rq", "Calidad regulatoria", "gobernanza", "estimador -2.5 a 2.5", False, "cobertura 100% pero redundante con ge (Spearman 0.924); se prioriza ge"),
    ("WGI", "rl", "Estado de derecho", "gobernanza", "estimador -2.5 a 2.5", True, "cobertura 100%; preferida sobre cc (Spearman 0.937 entre ambas)"),
    ("WGI", "cc", "Control de la corrupcion", "gobernanza", "estimador -2.5 a 2.5", False, "cobertura 100% pero redundante con rl (Spearman 0.937) y con la variable de corrupcion que ya aporta WHR"),
]

# DataFrame con la fecha de la decision agregada aca (no en cada tupla, para no repetirla 30 veces)
from pyspark.sql import functions as F
df_wb = spark.createDataFrame(filas_wb, ["fuente", "codigo", "nombre", "tema", "unidad", "incluido", "motivo"]) \
             .withColumn("fecha_decision", F.lit("2026-11-10").cast("date"))

df_wb.createOrReplaceTempView("fuente_datos_wb")

spark.sql("""
MERGE INTO happiness.meta.config_indicadores AS destino
USING fuente_datos_wb
ON destino.fuente = fuente_datos_wb.fuente AND destino.codigo = fuente_datos_wb.codigo
WHEN NOT MATCHED THEN INSERT *
""")

# Verificacion: cuantos quedaron incluidos por fuente
display(spark.sql("""
    SELECT fuente, incluido, COUNT(*) AS cantidad
    FROM happiness.meta.config_indicadores
    GROUP BY fuente, incluido
    ORDER BY fuente, incluido DESC
"""))